# Qiskit Lektion 1: Vom Statevector zur Messung

**Ziel:** Du verstehst, was hinter deinen beiden funktionierenden Zellen (Hadamard und Bell-Zustand) steckt, und lernst den Schritt vom *Zustandsvektor* zur *Messung*.

## Lernpfad (Überblick)

| Lektion | Thema | Kernfrage |
|---|---|---|
| **1 (hier)** | Qubit, Superposition, Verschränkung, Messung | Was ist der Zustand, und was sehe ich beim Messen? |
| 2 | Weitere Gatter (X, Y, Z, S, T), Bloch-Kugel | Wie drehe ich einen Zustand gezielt? |
| 3 | Mehrere Qubits, Phase, Interferenz | Warum ist Interferenz die Quelle des Quantenvorteils? |
| 4 | Algorithmen: Deutsch-Jozsa, Grover | Wo schlägt Quanten klassisch? |
| 5 | Variationale Circuits (VQC) | Brücke zu Machine Learning |

## 1. Setup und Rückblick: Was du schon hast

Deine beiden Zellen berechnen den Zustand **exakt** (ohne Zufall), mit `Statevector`. Hier dieselbe Idee, mit Kommentaren.

In [1]:
# Importe: QuantumCircuit baut den Schaltkreis, Statevector berechnet den exakten Zustand
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# --- Ein Qubit: Hadamard-Gatter erzeugt eine Superposition ---
qc1 = QuantumCircuit(1)   # Circuit mit 1 Qubit, Startzustand |0>
qc1.h(0)                  # Hadamard auf Qubit 0: |0> -> (|0> + |1>)/sqrt(2)

state1 = Statevector(qc1) # exakten Zustandsvektor berechnen (keine Messung, kein Zufall)
print(qc1.draw())         # Schaltkreis als Text-Diagramm
print(state1)             # Amplituden: beide 0.7071 = 1/sqrt(2)

ImportError: cannot import name '_accelerate' from partially initialized module 'qiskit' (most likely due to a circular import) (/home/sabir/quantum_lstm_project/qiskit/__init__.py)

## 2. Theorie: Definitionen und Gleichungen

**Definition (Qubit).** Ein Qubit ist ein normierter Vektor in $\mathbb{C}^2$:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad \alpha,\beta\in\mathbb{C}, \quad |\alpha|^2+|\beta|^2 = 1$$

mit $|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}$ und $|1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$.

**Definition (Gatter).** Ein Gatter ist eine **unitäre Matrix** $U$ mit $U^\dagger U = \mathbb{1}$. Das erhält die Norm.

| Gatter | Matrix | Wirkung | Qiskit |
|---|---|---|---|
| Hadamard $H$ | $\frac{1}{\sqrt2}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ | $\lvert0\rangle\to\frac{\lvert0\rangle+\lvert1\rangle}{\sqrt2}$ | `qc.h(i)` |
| Pauli-$X$ | $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ | Bit-Flip $\lvert0\rangle\leftrightarrow\lvert1\rangle$ | `qc.x(i)` |
| Pauli-$Z$ | $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ | Phase: $\lvert1\rangle\to-\lvert1\rangle$ | `qc.z(i)` |
| CNOT | $\lvert0\rangle\langle0\rvert\otimes\mathbb 1+\lvert1\rangle\langle1\rvert\otimes X$ | flippt Ziel, wenn Kontrolle $=1$ | `qc.cx(k, z)` |

**Bell-Zustand.** Aus $H$ auf Qubit 0 und anschließend CNOT:

$$|00\rangle \xrightarrow{H\otimes\mathbb 1} \tfrac{1}{\sqrt2}(|00\rangle+|10\rangle) \xrightarrow{\text{CNOT}} \tfrac{1}{\sqrt2}(|00\rangle+|11\rangle) = |\Phi^+\rangle$$

Dieser Zustand lässt sich **nicht** als Produkt $|\psi_1\rangle\otimes|\psi_2\rangle$ schreiben. Das ist die Definition von **Verschränkung**.

**Born-Regel.** Die Wahrscheinlichkeit, beim Messen das Ergebnis $x$ zu erhalten, ist $p(x)=|\langle x|\psi\rangle|^2$.

> **Achtung, Qiskit-Konvention (little-endian):** Qubit 0 steht in Bitstrings **rechts**. Der Zustand $|q_1 q_0\rangle$ mit $q_0=1$, $q_1=0$ heißt in Qiskit `'01'`.

## 3. Gatter als Matrizen prüfen

Als Physiker kannst du die Matrizen direkt nachrechnen. So siehst du, dass `h(0)` wirklich die Hadamard-Matrix ist.

In [ ]:
import numpy as np
from qiskit.quantum_info import Operator

# Operator(qc) liefert die unitäre Matrix des gesamten Circuits
H = Operator(qc1).data
print("Hadamard-Matrix aus Qiskit:\n", np.round(H, 4))

# Kontrolle: Ist H unitär? H^dagger @ H muss die Einheitsmatrix ergeben
print("\nH^† H =\n", np.round(H.conj().T @ H, 4))

# Kontrolle: H wendet man zweimal an -> Identität (H ist sein eigenes Inverses)
print("\nH @ H =\n", np.round(H @ H, 4))

## 4. Der Bell-Zustand mit Kommentaren

In [ ]:
# Circuit mit 2 Qubits, beide starten in |0>
qc2 = QuantumCircuit(2)

qc2.h(0)       # Schritt 1: Superposition auf Qubit 0
qc2.cx(0, 1)   # Schritt 2: CNOT, Qubit 0 = Kontrolle, Qubit 1 = Ziel -> Verschränkung

print(qc2.draw())          # Diagramm anzeigen

state2 = Statevector(qc2)  # exakter Zustand
print(state2)              # Amplituden: |00> und |11> je 0.7071, sonst 0

## 5. Von Amplituden zu Wahrscheinlichkeiten

Amplituden sind komplex und nicht direkt messbar. Messbar sind nur die **Wahrscheinlichkeiten** $|\text{Amplitude}|^2$.

In [ ]:
# probabilities_dict() wendet die Born-Regel an: p(x) = |Amplitude|^2
probs = {str(k): round(float(v), 4) for k, v in state2.probabilities_dict().items()}  # sauber formatiert
print("Wahrscheinlichkeiten:", probs)   # {'00': 0.5, '11': 0.5}

# Zur Kontrolle von Hand: Betragsquadrat der Amplituden
amplituden = state2.data
print("Betragsquadrate:", np.round(np.abs(amplituden) ** 2, 4))

## 6. Messung: Was ein echtes Experiment liefert

Ein Statevector ist ein **Rechenwerkzeug**. Auf echter Hardware gibt es ihn nicht. Dort misst man viele Male (*shots*) und erhält Häufigkeiten.

Wichtig: Für die Messung braucht der Circuit **klassische Bits**. `measure_all()` fügt sie hinzu und misst alle Qubits.

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator

# Bell-Circuit neu aufbauen, diesmal MIT Messung am Ende
qc_meas = QuantumCircuit(2)
qc_meas.h(0)
qc_meas.cx(0, 1)
qc_meas.measure_all()      # fügt klassische Bits hinzu und misst beide Qubits

sim = AerSimulator()                       # Standard-Simulator (Stichproben-Modus)
compiled = transpile(qc_meas, sim)         # auf den Simulator zuschneiden

# shots=1024: der Circuit wird 1024-mal ausgeführt, jedes Mal ein zufälliges Ergebnis
result = sim.run(compiled, shots=1024).result()
counts = result.get_counts()               # Häufigkeiten der Ergebnisse
print(counts)                              # z. B. {'00': 512, '11': 512}, nie '01' oder '10'

In [ ]:
# Histogramm der Ergebnisse (benötigt matplotlib)
from qiskit.visualization import plot_histogram
plot_histogram(counts)

**Beobachtung.** Nur `00` und `11` treten auf, jeweils etwa 50 %. Jedes einzelne Ergebnis ist zufällig, aber die beiden Qubits sind **perfekt korreliert**. Das ist die Signatur der Verschränkung.

### Rückblick: Warum dein erster Simulator-Code scheiterte

| Ansatz | Ergebnis | Wann sinnvoll |
|---|---|---|
| `Statevector(qc)` | exakter Zustand, ohne Messung | Lernen, Debuggen, kleine Circuits |
| `AerSimulator` + `measure_all()` + `get_counts()` | Häufigkeiten, wie auf echter Hardware | Realistische Experimente, Rauschmodelle |
| `AerSimulator` + `save_statevector()` + `get_statevector()` | Zustand aus dem Simulator | Große Circuits, Rauschsimulation |

## 7. Übungen
Versuche jede Aufgabe zuerst selbst, dann schau in die Lösung darunter.

**Aufgabe 1.** Welchen Zustand erzeugt `X` auf $|0\rangle$? Prüfe mit `Statevector`.

**Aufgabe 2.** Baue den Zustand $|\Psi^+\rangle=\frac{1}{\sqrt2}(|01\rangle+|10\rangle)$.

**Aufgabe 3.** Wende auf $|0\rangle$ zuerst `H`, dann `Z` an. Welchen Zustand erhältst du? Welche Wahrscheinlichkeiten misst man?

**Aufgabe 4 (Theorie).** Zeige $H^2=\mathbb 1$ durch Ausmultiplizieren der Matrix.

In [ ]:
# Aufgabe 1: hier dein Code

In [ ]:
# Aufgabe 2: hier dein Code

In [ ]:
# Aufgabe 3: hier dein Code

## 8. Lösungen

### Lösung 1
$X|0\rangle=|1\rangle$. Der Statevector ist `[0, 1]`.

In [ ]:
qa = QuantumCircuit(1)
qa.x(0)                    # Bit-Flip: |0> -> |1>
print(Statevector(qa))     # Amplitude 1 auf Index 1, also |1>

### Lösung 2
Startpunkt ist der Bell-Zustand $|\Phi^+\rangle$. Ein zusätzliches `X` auf Qubit 1 vertauscht dort $0\leftrightarrow1$:
$|00\rangle\to|10\rangle$ und $|11\rangle\to|01\rangle$. Das Ergebnis ist $|\Psi^+\rangle$.

In [ ]:
qb = QuantumCircuit(2)
qb.h(0)          # Superposition
qb.cx(0, 1)      # Bell-Zustand |Phi+> = (|00> + |11>)/sqrt(2)
qb.x(1)          # Qubit 1 flippen -> (|10> + |01>)/sqrt(2) = |Psi+>
print(Statevector(qb))   # Amplituden auf Index 1 ('01') und Index 2 ('10')

### Lösung 3
$Z$ ändert das Vorzeichen von $|1\rangle$:
$ZH|0\rangle=\frac{1}{\sqrt2}(|0\rangle-|1\rangle)=|-\rangle$.
Die Wahrscheinlichkeiten sind trotzdem $50\,\%/50\,\%$, denn $|-1/\sqrt2|^2=1/2$.
**Lerneffekt:** Die *Phase* (das Vorzeichen) ist bei einer direkten Messung unsichtbar. Sie wirkt erst durch Interferenz. Das ist das Thema von Lektion 3.

In [ ]:
qc_ = QuantumCircuit(1)
qc_.h(0)                       # (|0> + |1>)/sqrt(2)
qc_.z(0)                       # Phase: (|0> - |1>)/sqrt(2)
zst = Statevector(qc_)
print(zst)                     # Amplituden: +0.7071 und -0.7071
print({str(k): round(float(v), 4) for k, v in zst.probabilities_dict().items()})  # trotzdem {'0': 0.5, '1': 0.5}

### Lösung 4
$$H^2=\frac12\begin{pmatrix}1&1\\1&-1\end{pmatrix}\begin{pmatrix}1&1\\1&-1\end{pmatrix}
=\frac12\begin{pmatrix}1+1&1-1\\1-1&1+1\end{pmatrix}=\begin{pmatrix}1&0\\0&1\end{pmatrix}=\mathbb 1$$

Das siehst du auch in Abschnitt 3 in der Ausgabe von `H @ H`.

## 9. Zusammenfassung

- Ein Qubit ist ein normierter Vektor in $\mathbb{C}^2$, ein Gatter eine unitäre Matrix.
- `Statevector(qc)` berechnet den exakten Zustand, nur ohne Messungen im Circuit.
- Messen heißt: Born-Regel, zufälliges Ergebnis, viele *shots* ergeben Häufigkeiten.
- Verschränkung zeigt sich als perfekte Korrelation der Messergebnisse.
- Phasen sind bei direkter Messung unsichtbar. Das führt zu Lektion 3.

**Nächster Schritt:** Lektion 2 mit Pauli-Gattern, Rotationen und der Bloch-Kugel.